# 🧪 NutriTrack — Complete Test Suite

Runs all test scripts directly by importing functions from the `app/tests` directory.
- Output metrics are gathered into specific DataFrames.
- Raw data responses (p1-p4, q1-q6) are saved to `app/data/tests` for deep inspection.
- API Server is briefly spawned inside the notebook to run API endpoint tests.

## 0. Setup & Imports

In [1]:
import sys, os, time, json
import pandas as pd
from IPython.display import display

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, "config", ".env"))

import subprocess
import requests
import time
import tests.test_api as api_tests

FAST_FOOD_IMG = os.path.join(project_root, "..", "data", "images", "food", "fast_food.jpg")
COM_TAM_IMG   = os.path.join(project_root, "..", "data", "images", "food", "com_tam.jpg")

TEST_OUTPUT_DIR = os.path.join(project_root, "data", "tests")
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

def save_raw_output(filename: str, data: dict|list):
    if not data: return
    path = os.path.join(TEST_OUTPUT_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved {filename}")

print(f"✅ Setup complete. Outputs will be saved to: {TEST_OUTPUT_DIR}")

✅ Setup complete. Outputs will be saved to: d:\Project\Code\nutritrack-documentation\app\data\tests


## 1. USDA Client Tests

In [2]:
from third_apis.USDA import USDAClient
import tests.test_usda_client as usda_tests

usda_client = USDAClient(api_key=os.getenv("USDA_API_KEY"))

2026-03-18 18:06:48 | INFO     | third_apis.USDA                | USDAClient initialized (api_key=***)


In [3]:
usda_results = usda_tests.run_all(usda_client)
print(f"\nUSDA Client tests passed: {sum(1 for r in usda_results if r)}/{len(usda_results)}")


─── USDA Client Tests ─────────────────────────────────────────────────

  ─────[NUTRITION TESTS]─────
    1. 'chicken breast': cal=165.0  pro=20.4  fat=8.1  carb=1.1 (✅)
    2. 'white rice cooked': cal=96.0  pro=2.0  fat=0.2  carb=21.0 (✅)
    3. 'egg fried': cal=196.0  pro=13.6  fat=14.8  carb=0.8 (✅)
    4. 'cơm tấm': cal=467.0  pro=10.0  fat=16.7  carb=66.7 (✅)
    4/4 passed ✅

  ─────[INGREDIENTS TEST]─────
    1. 'chocolate': 10 items: ['sugar', 'sunflower oil', 'coconut oil'] (✅)
    1/1 passed ✅

  ─────[NUTR+ING TEST]─────
    1. 'chicken breast': desc='chicken breast'  cal=165.0 (✅)
    1/1 passed ✅

  ─────[BY WEIGHT TEST]─────
    1. 'chicken breast' 150g: cal=247.5  pro=30.6  fat=12.2  carb=1.6 (✅)
    1/1 passed ✅

  ─────[CACHE L1 TEST]─────
    1. 'chicken breast': 0.0035s (✅)
    1/1 passed ✅

  ─────[CACHE STATS TEST]─────
    1. cache_stats(): L1=5/256  L2=1132 (expired=0) (✅)
    1/1 passed ✅

  ─────[NORMALIZE TESTS]─────
    1. 'Chicken Breast': → 'chicken breas

## 1.1 Avocavo Nutrition Client Tests

In [4]:
from third_apis.AvocavoNutrition import AvocavoNutritionClient
import tests.test_avocavonutrition_client as avocavo_tests

avocavo_client = AvocavoNutritionClient(api_key=os.getenv("AVOCAVO_NUTRITION_API_KEY", "DEMO_KEY"))

2026-03-18 18:08:08 | INFO     | third_apis.AvocavoNutrition    | AvocavoNutritionClient initialized (api_key=***)


In [5]:
avocavo_results = avocavo_tests.run_all(avocavo_client)
print(f"\nAvocavo Nutrition Client tests passed: {sum(1 for r in avocavo_results if r)}/{len(avocavo_results)}")


─── Avocavo Nutrition Client Tests ────────────────────────────────────

  ─────[CLIENT INIT TESTS]─────
    1. demo key: initialized with key='DEMO_KEY' (✅)
    2. empty key: initialized with key='None' (✅)
    3. none key: initialized with key='None' (✅)
    4. custom key: initialized with key='valid_key_123' (✅)
    4/4 passed ✅

  ─────[NUTRITION TESTS]─────
    1. 'chicken breast': cal=165.0  pro=31.0  fat=3.6  carb=0.0 (✅)
    2. 'white rice cooked': cal=130.0  pro=2.4  fat=0.2  carb=28.6 (✅)
    3. 'egg fried': cal=196.0  pro=13.6  fat=14.8  carb=0.8 (✅)
    4. 'cơm tấm': cal=112.0  pro=0.4  fat=0.1  carb=27.2 (✅)
    5. 'phở bò': cal=95.0  pro=2.5  fat=3.3  carb=13.7 (✅)
    5/5 passed ✅
2026-03-18 18:08:10 | ERROR    | third_apis.AvocavoNutrition    | Avocavo Nutrition API HTTP error: 400 Client Error: Bad Request for url: https://app.avocavo.app/api/v2/nutrition/ingredient (query='')
2026-03-18 18:08:10 | WARNING  | third_apis.AvocavoNutrition    | get_nutritions: no result 

## 1.2 OpenFoodFacts Client Tests

In [6]:
from third_apis.OpenFoodFacts import OpenFoodFactsClient
import tests.test_openfoodfacts_client as off_tests

off_client = OpenFoodFactsClient()

2026-03-18 17:44:57 | INFO     | third_apis.OpenFoodFacts       | OpenFoodFactsClient initialized (no API key required)


In [7]:
off_results = off_tests.run_all(off_client)
print(f"\nOpenFoodFacts Client tests passed: {sum(1 for r in off_results if r)}/{len(off_results)}")


─── OpenFoodFacts Client Tests ────────────────────────────────────────

  ─────[CLIENT INIT TESTS]─────
    1. no api key (default): initialized successfully (✅)
    2. custom api key: initialized successfully (✅)
    3. empty api key: initialized successfully (✅)
    3/3 passed ✅
2026-03-18 17:44:59 | WARNING  | third_apis.OpenFoodFacts       | search_by_barcode: invalid or empty barcode input='abc'
2026-03-18 17:44:59 | WARNING  | third_apis.OpenFoodFacts       | search_by_barcode: invalid or empty barcode input=''

  ─────[BARCODE TEST]─────
    1. '8934563138165':  (❌)
    2. 'abc': invalid barcode -> found=False (✅)
    3. '': invalid barcode -> found=False (✅)
    4. '  3017620422003  ':  (❌)
    5. '123-456-789':  (❌)
    6. '0000000000000': found=False name=N/A (✅)
    3/6 passed ❌

  ─────[BARCODE PARSE TEST]─────
    1. fixture barcode parse: Hao Hao Mi Tom Chua Cay (hot S... | cal=455.0 (✅)
    1/1 passed ✅
2026-03-18 17:44:59 | WARNING  | third_apis.OpenFoodFacts       | 

## 1.3 Barcode Pipeline Tests

In [8]:
import importlib
import tests.test_barcode_pipeline as barcode_pipeline_tests

importlib.reload(barcode_pipeline_tests)
barcode_results = barcode_pipeline_tests.run_all()
print(f"\nBarcode Pipeline tests passed: {sum(1 for r in barcode_results if r)}/{len(barcode_results)}")


─── Barcode Pipeline & USDA Client Tests ──────────────────────────────

  ─────[SCAN]─────
    1. scan image: decoded=8934563138165 (✅)
    1/1 passed, 0 skipped ✅


2026-03-18 17:45:43 | ERROR    | tests.test_barcode_pipeline    | _test_lookup_barcode_from_cache failed: 
Traceback (most recent call last):
  File "d:\Project\Code\nutritrack-documentation\app\tests\test_barcode_pipeline.py", line 111, in _test_lookup_barcode_from_cache
    assert result.get("barcode") == barcode
AssertionError

  ─────[LOOKUP CACHE]─────
    1. lookup cache:  (❌)
    0/1 passed, 0 skipped ❌
2026-03-18 17:45:43 | WARNING  | scripts.scan_barcode           | lookup_barcode: invalid or empty barcode input='abc'

  ─────[INVALID INPUT]─────
    1. invalid barcode: returned validation error (✅)
    1/1 passed, 0 skipped ✅

  ─────[L1 CACHE]─────
    1. L1 cache hit: second lookup returned from RAM cache (✅)
    1/1 passed, 0 skipped ✅

  ─────[PIPELINE]─────
    1. pipeline: found=True barcode=8934563138165 (✅)
    1/1 passed, 0 skipped ✅

  ─────[PIPELINE+CLIENTS]─────
    1. pipeline+clients: found=True source=openfoodfacts level=L2 (✅)
    1/1 passed, 0 skipped ✅

  ──

## 2. Qwen3VL Model Tests
Runs the 3 model inference methods directly against the test images and records metrics.

In [3]:
from models.QWEN3VL import Qwen3VL
import tests.test_qwen_client as qwen_tests

qwen = Qwen3VL()

                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║            Initializing Qwen3VL            ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-12 19:26:16 | INFO     | models.QWEN3VL                 | model=qwen.qwen3-vl-235b-a22b, region=us-east-1


In [ ]:
qwen_results = qwen_tests.run_all(qwen)
for i, r in enumerate(qwen_results):
    # Extract method slug like 'converse', 'tools', etc.
    method_slug = r['method'].lower().replace(' ', '_')
    save_raw_output(f"q{i+1}_{method_slug}_{r['image']}.json", r.get("raw_output"))



─── Qwen3VL Model Tests ───────────────────────────────────────────────


In [ ]:
qwen_results = qwen_results # Already defined

print(f"\nQwen3VL tests passed: {sum(1 for r in qwen_results if r.get('success', False))}/{len(qwen_results)}")
print("(Note: Instructor tests are expected to fail with bedrock JSON schema)")

df_qwen = pd.DataFrame(qwen_results)
cols = ['method', 'image', 'status', 'time_s', 'dishes', 'ingredients', 'bedrock_calls', 'usda_calls', 'usda_cache_hits', 'tool_rounds', 'notes']
cols = [c for c in cols if c in df_qwen.columns]

display(df_qwen[cols].fillna(''))

## 3. Pipeline Tests
Testing the high-level orchestrated pipeline flows.

In [ ]:
import tests.test_pipeline as pipeline_tests

try:
    if qwen:
        pass
except:
    from models.QWEN3VL import Qwen3VL
    import tests.test_qwen_client as qwen_tests

    qwen = Qwen3VL()

pipe_results = pipeline_tests.run_all(qwen, usda_client)
for i, r in enumerate(pipe_results):
    method_slug = r['method'].lower().replace(' ', '_')
    save_raw_output(f"p{i+1}_pipeline_{method_slug}_{r['image']}.json", r.get("raw_output"))

                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║            Initializing Qwen3VL            ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-12 20:11:20 | INFO     | models.QWEN3VL                 | model=qwen.qwen3-vl-235b-a22b, region=us-east-1

─── Pipeline Tests ───────────────────────────────────────────────────────


In [ ]:
pipe_results = pipe_results # Already defined
print(f"\nPipeline tests passed: {sum(1 for r in pipe_results if r.get('success', False))}/{len(pipe_results)}")

df_pipe = pd.DataFrame(pipe_results)
pipe_cols = ['method', 'image', 'status', 'time_s', 'dishes', 'ingredients', 'bedrock_calls', 'usda_calls', 'usda_cache_hits', 'token_input', 'price_input', 'token_output', 'price_output', 'notes']
pipe_cols = [c for c in pipe_cols if c in df_pipe.columns]

df_results = df_pipe[pipe_cols].fillna('') 

output_dir = '../data/tests'
os.makedirs(output_dir, exist_ok=True)
df_results.to_csv(os.path.join(output_dir, 'df_results.csv'), index=False)

display(df_results)


Pipeline tests passed: 4/4


,method,image,status,time_s,dishes,ingredients,bedrock_calls,usda_calls,usda_cache_hits,token_input,price_input,token_output,price_output,notes
0,tools,com_tam,pass,40.90,1,7,0,6,5,7208,0.0252,1263,0.0177,"1 dish(es), 7 ingredient(s)"
1,manual,com_tam,pass,63.74,1,12,0,7,11,1250,0.0044,1463,0.0205,"1 dish(es), 12 ingredient(s)"
2,tools,fast_food,pass,136.68,11,37,0,11,18,9273,0.0325,5961,0.0835,"11 dish(es), 37 ingredient(s)"
3,manual,fast_food,pass,134.26,12,45,0,26,29,1517,0.0053,6280,0.0879,"12 dish(es), 45 ingredient(s)"


## 4. Label Analyzer Tests
Testing the nutrition label OCR pipeline with label and non-label images.

In [2]:
import tests.test_label_analyzer as label_tests

try:
    if qwen:
        pass
except:
    from models.QWEN3VL import Qwen3VL
    qwen = Qwen3VL()

label_results = label_tests.run_all(qwen)
for i, r in enumerate(label_results):
    save_raw_output(f"l{i+1}_label_{r['image']}.json", r.get("raw_output"))

                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║            Initializing Qwen3VL            ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-12 20:11:43 | INFO     | models.QWEN3VL                 | model=qwen.qwen3-vl-235b-a22b, region=us-east-1

─── Label Analyzer Tests ─────────────────────────────────────────────────
2026-03-12 20:11:58 | WARNING  | scripts.label_analyzer         | No nutrition label detected in image

  ─────[LABEL OCR TESTS]─────
    1. hao_hao: Detected 1 product(s)  [12.99s] (✅)
    2. com_tam: Correctly returned empty dishes for non-label image  [0.99s] (✅)
    2/2 passed ✅

───────────────────────────────────────────────────────────────────────
  2/2 passed ✅

Saved l1_label_hao_hao.json
Saved

In [ ]:
print(f"\nLabel analyzer tests passed: {sum(1 for r in label_results if r.get('success', False))}/{len(label_results)}")

df_label = pd.DataFrame(label_results)
label_cols = ['method', 'image', 'status', 'time_s', 'dishes', 'ingredients', 'notes']
label_cols = [c for c in label_cols if c in df_label.columns]

display(df_label[label_cols].fillna(''))


Label analyzer tests passed: 2/2


,method,image,status,time_s,dishes,ingredients,notes
0,label_ocr,hao_hao,pass,6.16,1,1,Detected 1 product(s)
1,label_ocr,com_tam,pass,1.03,0,0,Correctly returned empty dishes for non-label ...


## 5. API Endpoint Tests
Automatically spawn the Uvicorn server, hit the endpoints, and close the server.

In [ ]:
try:
    # Try hit healthcheck to confirm up
    host = "http://localhost:8000/health"
    resp = requests.get(host, timeout=5)
    print("Server is UP.")
    
    api_results = api_tests.run_all()
    print(f"\nAPI tests passed: {sum(1 for r in api_results if r.get('success'))}/{len(api_results)}")
    
except Exception as e:
    print(f"Failed during API tests: {e}")


Server is UP.

─── API Endpoint Tests ───────────────────────────────────────────────────

  ─────[HEALTH TEST]─────
    1. /health: Health check OK  [HTTP 200] (✅)
    1/1 passed ✅

  ─────[ROOT TEST]─────
    1. /: Root OK — qwen=qwen.qwen3-vl-235b-a22b, usda=True  [HTTP 200] (✅)
    1/1 passed ✅
